# Larger Random-Head LoRA Evaluation

Eval-only notebook for the reviewer control.

This notebook does **not retrain** either adapter. It loads:

- baseline LLaVA-1.5-7B
- targeted-head LoRA adapter: `results/stage2_lora_adapter`
- random-head LoRA adapter: `results/stage2_random_heads32_lora_adapter`

Then it evaluates all arms on the same larger image split and reports CHAIR plus average caption length. Start with `EVAL_N = 200`; switch to 400 if runtime allows.


## 0. Install dependencies

Run once, then Runtime -> Restart session. After restart, skip this cell and start from imports.


In [1]:
# RUN ONCE, then Runtime -> Restart session. Do not run imports before restarting.
!nvidia-smi --query-gpu=name,memory.total --format=csv

# Colab sometimes ships with NumPy 2.x while compiled deps expect NumPy 1.x.
!pip install -q --no-cache-dir --force-reinstall "numpy==1.26.4"
!pip install -q "transformers>=4.47" "accelerate>=0.33" "tokenizers>=0.21"
!pip install -q peft bitsandbytes
!pip install -q pillow tqdm pycocotools spacy sentencepiece
!python -m spacy download en_core_web_sm -q

print('Done. Now Runtime -> Restart session, then skip this cell and start from imports.')


name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 146.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 248.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, b

## 1. Imports and config

Set `WORK_DIR` to your Drive folder. For your screenshot, this should be `/content/drive/MyDrive/reducing_hallucinations` after adding the shared folder shortcut to My Drive.


In [1]:
import os, json, gc, random, re
from pathlib import Path
from collections import defaultdict

import torch
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

from google.colab import drive
drive.mount('/content/drive')

# ---- edit only this block if needed ----
WORK_DIR = '/content/drive/MyDrive/reducing_hallucinations'
EVAL_N = 200          # Recommended first run. Change to 400 if runtime allows.
MAX_NEW_TOKENS = 80   # Match the main Stage 2/Stage 4 evaluation budget.
REPETITION_PENALTY = 1.2  # Match validation_Experiments.ipynb gen_greedy.
LORA_SCALE = 0.75      # Control eval scale; use same value for targeted and random adapters.
EVAL_SOURCE = 'stage4_400img_results'  # Uses the same 400-image list if present.
RUN_POPE = False      # CHAIR+length is the key control; POPE is optional and slower.
# ---------------------------------------

COCO_DIR = f'{WORK_DIR}/coco'
RESULTS_DIR = f'{WORK_DIR}/results'
TARGETED_ADAPTER_DIR = f'{RESULTS_DIR}/stage2_lora_adapter'
RANDOM_ADAPTER_DIR = f'{RESULTS_DIR}/stage2_random_heads32_lora_adapter'
OUT_PATH = f'{RESULTS_DIR}/stage2_random_heads32_large_eval_n{EVAL_N}.json'

for path in [WORK_DIR, COCO_DIR, RESULTS_DIR, TARGETED_ADAPTER_DIR, RANDOM_ADAPTER_DIR]:
    print(path, 'OK' if os.path.exists(path) else 'MISSING')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
if device == 'cpu':
    raise RuntimeError('No GPU detected. Use Runtime -> Change runtime type -> GPU, then restart.')
print('GPU:', torch.cuda.get_device_name(0))
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


Mounted at /content/drive
/content/drive/MyDrive/reducing_hallucinations OK
/content/drive/MyDrive/reducing_hallucinations/coco OK
/content/drive/MyDrive/reducing_hallucinations/results OK
/content/drive/MyDrive/reducing_hallucinations/results/stage2_lora_adapter OK
/content/drive/MyDrive/reducing_hallucinations/results/stage2_random_heads32_lora_adapter OK
Device: cuda
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


## 2. Load COCO metadata and the larger evaluation split

The preferred split is the image order from `results/stage4_400img_results.json`, so the numbers anchor to a split already used in the paper. If that file is absent, the notebook falls back to `cache/selected_imgs.json`.


In [2]:
from pycocotools.coco import COCO

ann_path = f'{COCO_DIR}/annotations/instances_val2014.json'
if not os.path.exists(ann_path):
    raise FileNotFoundError(f'Missing COCO instances file: {ann_path}')

coco = COCO(ann_path)
cat_id_to_name = {c['id']: c['name'].lower() for c in coco.loadCats(coco.getCatIds())}
ALL_COCO_OBJECTS = set(cat_id_to_name.values())

stage4_path = f'{RESULTS_DIR}/stage4_400img_results.json'
selected_path = f'{WORK_DIR}/cache/selected_imgs.json'

if os.path.exists(stage4_path):
    with open(stage4_path) as f:
        stage4 = json.load(f)
    all_eval_ids = [int(x['img_id']) for x in stage4.get('eval_captions', [])]
    split_source = stage4_path
elif os.path.exists(selected_path):
    with open(selected_path) as f:
        selected = json.load(f)
    all_eval_ids = [int(x) for x in selected['ids']]
    split_source = selected_path
else:
    raise FileNotFoundError('Need either results/stage4_400img_results.json or cache/selected_imgs.json')

# Keep order stable and remove accidental duplicates.
seen = set()
all_eval_ids = [x for x in all_eval_ids if not (x in seen or seen.add(x))]
requested_eval_ids = all_eval_ids[:EVAL_N]
print(f'Split source: {split_source}')
print(f'Requested eval images: {len(requested_eval_ids)} / available {len(all_eval_ids)}')

# Resolve paths. Support both val2014_subset and full val2014 folder names.
img_dirs = [
    f'{COCO_DIR}/val2014_subset',
    f'{COCO_DIR}/val2014',
    f'{COCO_DIR}/images/val2014',
]

img_id_to_path = {}
for meta in coco.loadImgs(requested_eval_ids):
    found = None
    for d in img_dirs:
        candidate = f"{d}/{meta['file_name']}"
        if os.path.exists(candidate):
            found = candidate
            break
    if found is not None:
        img_id_to_path[int(meta['id'])] = found

missing = [img_id for img_id in requested_eval_ids if img_id not in img_id_to_path]
if missing:
    print(f'WARNING: missing {len(missing)} image files. First few: {missing[:10]}')

eval_images = [img_id for img_id in requested_eval_ids if img_id in img_id_to_path]
if len(eval_images) == 0:
    raise RuntimeError('No eval images found on disk. Check the COCO image folder.')

# Ground-truth COCO object categories per image.
def gt_objects_for_image(img_id):
    ann_ids = coco.getAnnIds(imgIds=[int(img_id)])
    anns = coco.loadAnns(ann_ids)
    return set(cat_id_to_name[a['category_id']] for a in anns if a.get('iscrowd', 0) == 0)

eval_gt_objects = [gt_objects_for_image(img_id) for img_id in eval_images]
print(f'Usable eval images: {len(eval_images)}')
print('First eval ids:', eval_images[:10])


loading annotations into memory...
Done (t=10.17s)
creating index...
index created!
Split source: /content/drive/MyDrive/reducing_hallucinations/results/stage4_400img_results.json
Requested eval images: 200 / available 400
Usable eval images: 200
First eval ids: [293474, 465878, 419401, 432962, 183519, 152771, 337826, 143572, 162358, 262626]


## 3. Load LLaVA-1.5-7B in 4-bit


In [3]:
from transformers import AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig

MODEL_ID = 'llava-hf/llava-1.5-7b-hf'
PROMPT_TEMPLATE = 'USER: <image>\nDescribe this image in detail.\nASSISTANT:'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    low_cpu_mem_usage=True,
    device_map={'': 0},
)
model.eval()
print(f'Model loaded. VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:134: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


processor_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/674 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

Model loaded. VRAM used: 4.07 GB


## 4. CHAIR scorer with caption length

This mirrors the Stage 2 CHAIR scorer but also records average word length and generated captions.


In [4]:
import spacy
nlp = spacy.load('en_core_web_sm')

COCO_SYNONYMS = {
    'person':        ['man','woman','people','boy','girl','child','guy','lady','kid','baby','player','rider','skier','surfer','snowboarder'],
    'car':           ['vehicle','automobile','sedan','suv'],
    'dog':           ['puppy','dogs'],
    'cat':           ['kitten','cats'],
    'tv':            ['television','monitor','screen'],
    'couch':         ['sofa'],
    'cell phone':    ['phone','cellphone','smartphone'],
    'dining table':  ['table','desk'],
    'wine glass':    ['glass'],
    'bicycle':       ['bike'],
    'motorcycle':    ['motorbike'],
    'airplane':      ['plane','jet'],
    'potted plant':  ['plant'],
    'laptop':        ['computer'],
    'refrigerator':  ['fridge'],
    'truck':         ['lorry'],
    'boat':          ['ship','sailboat'],
    'fire hydrant':  ['hydrant'],
    'hot dog':       ['hotdog'],
    'traffic light': ['stoplight'],
    'sports ball':   ['ball','football','soccer ball','basketball'],
    'baseball bat':  ['bat'],
    'tennis racket': ['racket','racquet'],
}
MULTIWORD_ALIASES = {
    'hydrant': 'fire hydrant',
    'hotdog': 'hot dog',
    'stoplight': 'traffic light',
    'bat': 'baseball bat',
    'racket': 'tennis racket',
    'racquet': 'tennis racket',
}
OBJECT_VOCAB = set(ALL_COCO_OBJECTS)
for syns in COCO_SYNONYMS.values():
    OBJECT_VOCAB.update(syns)
OBJECT_VOCAB.update(MULTIWORD_ALIASES.keys())


def clean_caption(text):
    return re.sub(r'\s+', ' ', text).strip()

@torch.no_grad()
def generate_caption(model_obj, image_path, max_new_tokens=MAX_NEW_TOKENS):
    img = Image.open(image_path).convert('RGB')
    inputs = processor(text=PROMPT_TEMPLATE, images=img, return_tensors='pt').to(device, torch.float16)
    inputs['input_ids'] = inputs['input_ids'].long()
    inputs['attention_mask'] = inputs['attention_mask'].long()
    out = model_obj.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        repetition_penalty=REPETITION_PENALTY,
        return_dict_in_generate=True,
        use_cache=True,
    )
    gen_ids = out.sequences[0, inputs['input_ids'].shape[1]:]
    caption = processor.tokenizer.decode(gen_ids, skip_special_tokens=True)
    return clean_caption(caption), gen_ids.cpu()


def find_content_words(gen_ids, gt_objects):
    full_text = processor.tokenizer.decode(gen_ids, skip_special_tokens=True)
    gt_norm = set(o.lower() for o in gt_objects)
    expanded_gt = set(gt_norm)
    for canonical, syns in COCO_SYNONYMS.items():
        if canonical in gt_norm:
            expanded_gt.update(syns)
    for alias, canonical in MULTIWORD_ALIASES.items():
        if canonical in gt_norm:
            expanded_gt.add(alias)

    doc = nlp(full_text)
    nouns = [tok.text.lower().strip() for tok in doc
             if tok.pos_ in ('NOUN', 'PROPN', 'ADJ') and len(tok.text.strip()) >= 2]
    if not nouns:
        return []

    accumulated = ''
    results = []
    noun_idx = 0
    for tok_i, tid in enumerate(gen_ids):
        if noun_idx >= len(nouns):
            break
        ts = processor.tokenizer.decode([int(tid)], skip_special_tokens=True).lower()
        accumulated += ts
        target = nouns[noun_idx]
        if target in accumulated:
            canonical = MULTIWORD_ALIASES.get(target, target)
            is_object = target in OBJECT_VOCAB
            is_hall = is_object and (target not in expanded_gt) and (canonical not in expanded_gt)
            results.append({'word': target, 'is_object': is_object, 'is_hallucinated': is_hall})
            cut = accumulated.rfind(target) + len(target)
            accumulated = accumulated[cut:]
            noun_idx += 1
    return results

@torch.no_grad()
def chair_eval_with_lengths(model_obj, label, images, gt_objects_list):
    model_obj.eval()
    chairs_list, chairi_list, word_lens, token_lens = [], [], [], []
    rows = []

    for img_id, gt_set in tqdm(list(zip(images, gt_objects_list)), total=len(images), desc=f'CHAIR {label}'):
        img_path = img_id_to_path.get(int(img_id))
        if not img_path or not os.path.exists(img_path):
            continue
        caption, gen_ids = generate_caption(model_obj, img_path)
        cw = find_content_words(gen_ids, gt_set)
        obj_words = [c for c in cw if c['is_object']]
        hall_words = [c for c in cw if c['is_hallucinated']]
        chairs = 1 if hall_words else 0
        chairi = len(hall_words) / max(len(obj_words), 1)
        word_len = len(caption.split())
        token_len = int(len(gen_ids))

        chairs_list.append(chairs)
        chairi_list.append(chairi)
        word_lens.append(word_len)
        token_lens.append(token_len)
        rows.append({
            'img_id': int(img_id),
            'gt_objects': sorted(gt_set),
            'caption': caption,
            'chair_s': chairs,
            'chair_i': chairi,
            'object_words': [c['word'] for c in obj_words],
            'hallucinated_words': [c['word'] for c in hall_words],
            'word_len': word_len,
            'token_len': token_len,
        })
        torch.cuda.empty_cache()

    return {
        'CHAIRs': float(np.mean(chairs_list)),
        'CHAIRi': float(np.mean(chairi_list)),
        'avg_len': float(np.mean(word_lens)),
        'avg_token_len': float(np.mean(token_lens)),
        'n': len(chairs_list),
        'captions': rows,
    }

print('Scorer ready.')


Scorer ready.


## 5. Evaluate baseline, targeted-head LoRA, and random-head LoRA

This is the main cell. It can take a while because it generates captions for three arms.


In [5]:
from peft import PeftModel


def set_lora_scale(model_obj, scale):
    n_scaled = 0
    for name, module in model_obj.named_modules():
        if hasattr(module, 'scaling') and isinstance(module.scaling, dict):
            for key in module.scaling:
                module.scaling[key] = scale
            n_scaled += 1
    print(f'LoRA scaling set to {scale} on {n_scaled} modules')


results = {
    'config': {
        'work_dir': WORK_DIR,
        'eval_n_requested': EVAL_N,
        'eval_n_actual': len(eval_images),
        'max_new_tokens': MAX_NEW_TOKENS,
        'eval_source': split_source,
        'targeted_adapter_dir': TARGETED_ADAPTER_DIR,
        'random_adapter_dir': RANDOM_ADAPTER_DIR,
    },
    'chair': {},
}

print('=== Baseline ===')
results['chair']['baseline'] = chair_eval_with_lengths(model, 'baseline', eval_images, eval_gt_objects)
print({k: results['chair']['baseline'][k] for k in ['CHAIRs','CHAIRi','avg_len','n']})

print('\nLoading targeted-head adapter...')
peft_model = PeftModel.from_pretrained(model, TARGETED_ADAPTER_DIR, adapter_name='targeted')
peft_model.eval()
peft_model.set_adapter('targeted')
set_lora_scale(peft_model, LORA_SCALE)
print('=== Targeted-head LoRA ===')
results['chair']['targeted_lora'] = chair_eval_with_lengths(peft_model, 'targeted', eval_images, eval_gt_objects)
print({k: results['chair']['targeted_lora'][k] for k in ['CHAIRs','CHAIRi','avg_len','n']})

print('\nLoading random-head adapter into same PEFT model...')
peft_model.load_adapter(RANDOM_ADAPTER_DIR, adapter_name='random')
peft_model.set_adapter('random')
set_lora_scale(peft_model, LORA_SCALE)
print('=== Random-head LoRA ===')
results['chair']['random_heads32_lora'] = chair_eval_with_lengths(peft_model, 'random', eval_images, eval_gt_objects)
print({k: results['chair']['random_heads32_lora'][k] for k in ['CHAIRs','CHAIRi','avg_len','n']})

with open(OUT_PATH, 'w') as f:
    json.dump(results, f, indent=2)

print('\nSaved:', OUT_PATH)


=== Baseline ===


CHAIR baseline:   0%|          | 0/200 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


{'CHAIRs': 0.405, 'CHAIRi': 0.15446825396825395, 'avg_len': 60.395, 'n': 200}

Loading targeted-head adapter...
LoRA scaling set to 0.75 on 93 modules
=== Targeted-head LoRA ===


/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:622: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.model.vision_tower.encoder.layers.7.self_attn.k_proj.lora_A.targeted.weight', 'base_model.model.model.vision_tower.encoder.layers.7.self_attn.k_proj.lora_B.targeted.weight', 'base_model.model.model.vision_tower.encoder.layers.7.self_attn.v_proj.lora_A.targeted.weight', 'base_model.model.model.vision_tower.encoder.layers.7.self_attn.v_proj.lora_B.targeted.weight', 'base_model.model.model.vision_tower.encoder.layers.7.self_attn.q_proj.lora_A.targeted.weight', 'base_model.model.model.vision_tower.encoder.layers.7.self_attn.q_proj.lora_B.targeted.weight', 'base_model.model.model.vision_tower.encoder.layers.9.self_attn.k_proj.lora_A.targeted.weight', 'base_model.model.model.vision_tower.encoder.layers.9.self_attn.k_proj.lora_B.targeted.weight', 'base_model.model.model.vision_tower.encoder.layers.9.self_attn.v_proj.lora_A.targeted.weight', '

CHAIR targeted:   0%|          | 0/200 [00:00<?, ?it/s]

{'CHAIRs': 0.335, 'CHAIRi': 0.1317186147186147, 'avg_len': 40.355, 'n': 200}

Loading random-head adapter into same PEFT model...
LoRA scaling set to 0.75 on 96 modules
=== Random-head LoRA ===


CHAIR random:   0%|          | 0/200 [00:00<?, ?it/s]

{'CHAIRs': 0.405, 'CHAIRi': 0.15446825396825395, 'avg_len': 60.395, 'n': 200}

Saved: /content/drive/MyDrive/reducing_hallucinations/results/stage2_random_heads32_large_eval_n200.json


## 6. Summary table

Paste this output back into the chat so we can update the paper.


In [6]:
summary_rows = []
for name, res in results['chair'].items():
    summary_rows.append((name, res['CHAIRs'], res['CHAIRi'], res['avg_len'], res['avg_token_len'], res['n']))

print('=' * 86)
print(f'{"Method":<24} {"CHAIRs":>8} {"CHAIRi":>8} {"Avg words":>10} {"Avg tokens":>11} {"n":>6}')
print('-' * 86)
for name, chairs, chairi, avg_len, avg_tok, n in summary_rows:
    print(f'{name:<24} {chairs:>8.3f} {chairi:>8.3f} {avg_len:>10.1f} {avg_tok:>11.1f} {n:>6}')
print('=' * 86)
print('Result JSON:', OUT_PATH)

b = results['chair']['baseline']
t = results['chair']['targeted_lora']
r = results['chair']['random_heads32_lora']
print('\nPaper-ready comparison draft:')
print(
    f'On the shared n={b["n"]} evaluation split, baseline CHAIRs/CHAIRi were '
    f'{b["CHAIRs"]:.3f}/{b["CHAIRi"]:.3f} at {b["avg_len"]:.1f} words. '
    f'Targeted-head LoRA gave {t["CHAIRs"]:.3f}/{t["CHAIRi"]:.3f} at {t["avg_len"]:.1f} words, '
    f'while layer-matched random-head LoRA gave {r["CHAIRs"]:.3f}/{r["CHAIRi"]:.3f} at {r["avg_len"]:.1f} words.'
)


Method                     CHAIRs   CHAIRi  Avg words  Avg tokens      n
--------------------------------------------------------------------------------------
baseline                    0.405    0.154       60.4        75.6    200
targeted_lora               0.335    0.132       40.4        52.0    200
random_heads32_lora         0.405    0.154       60.4        75.6    200
Result JSON: /content/drive/MyDrive/reducing_hallucinations/results/stage2_random_heads32_large_eval_n200.json

Paper-ready comparison draft:
On the shared n=200 evaluation split, baseline CHAIRs/CHAIRi were 0.405/0.154 at 60.4 words. Targeted-head LoRA gave 0.335/0.132 at 40.4 words, while layer-matched random-head LoRA gave 0.405/0.154 at 60.4 words.
